In [ ]:
# ======================================================
# 0. HAFIZA VE ÇEKİRDEK (KERNEL) TEMİZLEYİCİ
# ======================================================
import os
import signal
import time

print("🧹 Arka planda asılı kalan tüm hafıza ve kütüphane kalıntıları temizleniyor...")
print("⚠️ DİKKAT: Birazdan ekranın sağ altında 'Oturum çöktü' veya 'Yeniden başlatıldı' yazısı çıkacak.")
print("✅ Bu tamamen NORMALDİR ve tam olarak yapmak istediğimiz şeydir!")

time.sleep(3)

# Mevcut Python sürecini (Kernel) zorla sonlandırarak Colab'i sıfırlamaya zorlar
os.kill(os.getpid(), signal.SIGKILL)

🧹 Arka planda asılı kalan tüm hafıza ve kütüphane kalıntıları temizleniyor...
⚠️ DİKKAT: Birazdan ekranın sağ altında 'Oturum çöktü' veya 'Yeniden başlatıldı' yazısı çıkacak.
✅ Bu tamamen NORMALDİR ve tam olarak yapmak istediğimiz şeydir!


In [1]:
# ======================================================
# 1. KÜTÜPHANELER
# ======================================================
print("\n📦 1/6: Kütüphaneler kuruluyor...")

import sys
import subprocess

print("🔄 Kütüphaneler en güncel ve uyumlu halleriyle kuruluyor (Bu işlem 1-2 dakika sürebilir)...")

# Tüm versiyon kilitlerini kaldırdık.
# Sistemin güncel PyTorch ve CUDA'sına en uygun versiyonlar otomatik seçilecek.
pip_install_cmd = [
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "transformers",
    "peft",
    "accelerate",
    "bitsandbytes",
    "trl",
    "datasets",
    "sentence-transformers",
    "chromadb",
    "fastapi",
    "uvicorn",
    "pyngrok",
    "nest-asyncio"
]

subprocess.run(pip_install_cmd, capture_output=False)

print("✅ Kütüphaneler başarıyla kuruldu!")


📦 1/6: Kütüphaneler kuruluyor...
🔄 Kütüphaneler en güncel ve uyumlu halleriyle kuruluyor (Bu işlem 1-2 dakika sürebilir)...
✅ Kütüphaneler başarıyla kuruldu!


In [2]:
# ======================================================
# 2. IMPORTLAR VE SİSTEM KONTROLÜ
# ======================================================
print("\n📦 2/6: Importlar yapılıyor...")

import os
# CUDA ve BitsAndBytes kütüphanelerinin birbirini bulması için path ayarı
if ":/usr/local/cuda/lib64" not in os.environ.get("LD_LIBRARY_PATH", ""):
    os.environ["LD_LIBRARY_PATH"] = os.environ.get("LD_LIBRARY_PATH", "") + ":/usr/local/cuda/lib64"

import torch
import trl
import transformers
import chromadb
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import Dataset
from sentence_transformers import SentenceTransformer

print("\n" + "=" * 40)
print("🖥️ SİSTEM BİLGİLERİ TEYİDİ")
print("=" * 40)

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ DİKKAT: GPU bulunamadı! Çalışma zamanı türünden GPU'yu seçtiğinize emin olun.")

print(f"✅ PyTorch Versiyonu: {torch.__version__}")
print(f"✅ CUDA Aktif mi: {torch.cuda.is_available()}")
print(f"✅ TRL Versiyonu: {trl.__version__}")
print(f"✅ Transformers Versiyonu: {transformers.__version__}")
print(f"✅ ChromaDB Versiyonu: {chromadb.__version__}")
print("=" * 40)
print("🚀 Import işlemi kusursuz tamamlandı! Bir sonraki adıma geçebilirsiniz.")


📦 2/6: Importlar yapılıyor...

🖥️ SİSTEM BİLGİLERİ TEYİDİ
✅ GPU: NVIDIA A100-SXM4-40GB
✅ VRAM: 42.4 GB
✅ PyTorch Versiyonu: 2.11.0+cu128
✅ CUDA Aktif mi: True
✅ TRL Versiyonu: 1.6.0
✅ Transformers Versiyonu: 5.12.0
✅ ChromaDB Versiyonu: 1.5.9
🚀 Import işlemi kusursuz tamamlandı! Bir sonraki adıma geçebilirsiniz.


In [3]:

# ======================================================
# 3. VERİYİ HAZIRLA
# ======================================================
print("\n📦 3/6: Veri hazırlanıyor...")
import os
import json
import shutil
import torch
from datasets import Dataset
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

data_path = "/content/enhanced_dataset.json"

if not os.path.exists(data_path):
    raise FileNotFoundError(f"❌ DİKKAT: '{data_path}' dosyası bulunamadı! Lütfen sol menüden JSON dosyanızı yükleyin.")

with open(data_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"✅ {len(data)} soru-cevap yüklendi")

USE_ALL_DATA = True
train_data = []

# Qwen2.5 Instruct formatına uygun Chat Template (System + User/Assistant)
for item in (data if USE_ALL_DATA else data[:100]):
    # Qwen formatına uygun şekilde metni formatlıyoruz
    text = f"<|im_start|>system\nSen E-YABANCI asistanısın.<|im_end|>\n<|im_start|>user\n{item['question']}<|im_end|>\n<|im_start|>assistant\n{item['answer']}<|im_end|>"
    train_data.append({"text": text})

if USE_ALL_DATA:
    print(f"✅ Tüm veri kullanılıyor: {len(train_data)} örnek")
else:
    print(f"✅ Test modu: {len(train_data)} örnek")

dataset = Dataset.from_list(train_data)

# ======================================================
# 4. CHROMADB (RAG HAZIRLIĞI)
# ======================================================
print("\n📦 4/6: ChromaDB hazırlanıyor...")

embedder = SentenceTransformer(
    "intfloat/multilingual-e5-large",
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

if os.path.exists("/content/chroma_db"):
    shutil.rmtree("/content/chroma_db")

client = chromadb.PersistentClient(path="/content/chroma_db")
collection = client.create_collection(
    name="e_yabanci",
    metadata={"hnsw:space": "cosine"}
)

answers  = [item['answer']   for item in data]
ids      = [str(item.get('id', i)) for i, item in enumerate(data)] # ID yoksa index kullan
questions= [item['question'] for item in data]
topics   = [item.get('topic', 'Genel') for item in data]

print("🔄 Embedding hesaplanıyor (E5 Modeli)...")
embeddings = embedder.encode(
    answers,
    normalize_embeddings=True,
    show_progress_bar=True
)

batch_size = 100
for i in range(0, len(ids), batch_size):
    end = min(i + batch_size, len(ids))
    collection.add(
        ids=ids[i:end],
        documents=answers[i:end],
        embeddings=embeddings[i:end].tolist(),
        metadatas=[
            {'id': ids[j], 'topic': topics[j], 'question': questions[j]}
            for j in range(i, end)
        ]
    )
    print(f"✅ {end}/{len(ids)} vektör veritabanına eklendi")

print(f"✅ ChromaDB hazır: Toplam {collection.count()} doküman indexlendi.")

# ======================================================
# 5. MODELİ YÜKLE (A100 OPTİMİZELİ)
# ======================================================
print("\n📦 5/6: Qwen2.5-7B Modeli yükleniyor... (Bu işlem boyut nedeniyle 2-3 dakika sürebilir)")

# A100 için özel bfloat16 optimizasyonu
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, # A100 performansı için bfloat16 kullanıldı
    bnb_4bit_use_double_quant=True
)

model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = "<|endoftext|>" # Qwen'in özel pad token'ı
    tokenizer.pad_token_id = tokenizer.convert_tokens_to_ids("<|endoftext|>")
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 # A100'e uygun ana veri tipi
)

model.config.use_cache = False
model.config.pretraining_tp = 1

print("✅ Model GPU'ya başarıyla yüklendi!")

# ======================================================
# 6. LORA (FINE-TUNING YAPILANDIRMASI)
# ======================================================
print("\n📦 6/6: LoRA yapılandırılıyor...")

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 4-bit model için gradient checkpointing hazırlığı
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)
model = get_peft_model(model, lora_config)

trainable = model.num_parameters(only_trainable=True)
total = model.num_parameters()
print(f"✅ Eğitilecek parametre: {trainable:,} / {total:,} (%{(trainable/total)*100:.2f})")
print("\n🚀 EĞİTİME (FINE-TUNING) GEÇMEYE HAZIRIZ!")


📦 3/6: Veri hazırlanıyor...
✅ 1000 soru-cevap yüklendi
✅ Tüm veri kullanılıyor: 1000 örnek

📦 4/6: ChromaDB hazırlanıyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Model GPU'ya başarıyla yüklendi!

📦 6/6: LoRA yapılandırılıyor...
✅ Eğitilecek parametre: 40,370,176 / 7,655,986,688 (%0.53)

🚀 EĞİTİME (FINE-TUNING) GEÇMEYE HAZIRIZ!


In [4]:
# ======================================================
# 7. FINE-TUNING (TRL 1.5+ VE A100 OPTİMİZELİ)
# ======================================================
print("\n🚀 Fine-tuning başlıyor! (A100'de ~15-20 dakika)")
print("=" * 60)

from trl import SFTConfig, SFTTrainer

# max_seq_length yerine yeni TRL mimarisinin beklediği max_length kullanıyoruz
training_args = SFTConfig(
    output_dir="/content/qwen-lora",
    per_device_train_batch_size=4,  # A100 için 4 idealdir
    gradient_accumulation_steps=2,
    num_train_epochs=3 if USE_ALL_DATA else 1,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    optim="paged_adamw_8bit",
    warmup_steps=30,
    lr_scheduler_type="cosine",
    max_length=1024,
    dataset_text_field="text",
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()

# ======================================================
# 8. MODELİ KAYDET
# ======================================================
print("\n✅ Fine-tuning tamamlandı! Model kaydediliyor...")

model.save_pretrained("/content/qwen-lora-final")
tokenizer.save_pretrained("/content/qwen-lora-final")
print("✅ Model /content/qwen-lora-final konumuna başarıyla kaydedildi")




🚀 Fine-tuning başlıyor! (A100'de ~15-20 dakika)


/tmp/ipykernel_8212/4264290464.py:10: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,4.440810
20,2.460043
30,1.415602
40,1.366992
50,1.373275
60,1.306520
70,1.301879
80,1.280596
90,1.331910
100,1.249024



✅ Fine-tuning tamamlandı! Model kaydediliyor...
✅ Model /content/qwen-lora-final konumuna başarıyla kaydedildi


In [5]:
# ======================================================
# 9. RAG + QWEN 2.5 + DEEP-TRANSLATOR (TAM ÇÖZÜM)
# ======================================================
print("\n📦 Çeviri kütüphanesi kuruluyor...")
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "deep-translator"], capture_output=False)

from deep_translator import GoogleTranslator
print("🔧 Çok Dilli RAG fonksiyonu oluşturuluyor...")

# Senin yazdığın dil tespit fonksiyonu
def detect_language(text: str) -> str:
    if any(c in text.lower() for c in "ğüşıöç"):
        return 'tr'
    if any('\u0600' <= c <= '\u06FF' for c in text):
        return 'ar'
    return 'en'

# Senin yazdığın çeviri fonksiyonu (source='auto' yapılarak daha güvenli hale getirildi)
def translate_text(text: str, target: str) -> str:
    if target == 'en':
        return text
    try:
        # Qwen ne üretirse üretsin, kullanıcının diline çevir
        return GoogleTranslator(source='auto', target=target).translate(text)
    except Exception as e:
        print(f"⚠️ Çeviri hatası: {e}")
        return text

# Modeli test modunda tutuyoruz
model.eval()

def rag_with_finetuned(question):
    # 1. Kullanıcının dilini tespit et
    user_lang = detect_language(question)

    # 2. RAG araması yap
    emb = embedder.encode(question, normalize_embeddings=True).tolist()
    results = collection.query(
        query_embeddings=[emb],
        n_results=3,
        include=["documents", "distances"]
    )

    docs   = results['documents'][0]
    scores = [1 - d for d in results['distances'][0]] if results['distances'][0] else []

    # Eğer bilgi yoksa kullanıcının dilinde "bilmiyorum" de
    if not docs or (scores and max(scores) < 0.80):
        no_info = {
            'tr': "Üzgünüm, bu konuda bilgim yok. Lütfen Türkiye ile ilgili başka bir soru sorunuz.",
            'en': "I'm sorry, I don't have information on this topic.",
            'ar': "آسف، ليس لدي معلومات حول هذا الموضوع."
        }
        return no_info.get(user_lang, no_info['en'])

    context = "\n\n".join(docs)

    # 3. Modele soruyu kendi rahat ettiği dilde (İngilizce veritabanı) soruyoruz
    prompt = f"<|im_start|>system\nYou are the E-YABANCI assistant. Answer the user's question clearly and concisely based ONLY on the provided INFORMATION.<|im_end|>\n<|im_start|>user\nINFORMATION:\n{context}\n\nQUESTION: {question}<|im_end|>\n<|im_start|>assistant\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id,
        )

    input_length = inputs['input_ids'].shape[1]
    raw_response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    raw_response = raw_response.replace("<|im_end|>", "").strip()

    # 4. Qwen'in verdiği ham cevabı kullanıcının diline çevir (Senin çözümün)
    final_response = translate_text(raw_response, user_lang)

    return final_response

# ======================================================
# 10. ÇOK DİLLİ TEST (DEEP-TRANSLATOR İLE)
# ======================================================
print("\n🧪 Çok Dilli RAG Testi yapılıyor...")

test_sorular = [
    "İstanbulkart nedir ve nasıl alabilirim?",
    "How do I open a bank account in Turkey as a foreigner?",
    "ما هي عقوبة تجاوز مدة التأشيرة في تركيا؟"
]

for soru in test_sorular:
    print(f"\n📨 Soru: {soru}")
    print(f"💬 Cevap: {rag_with_finetuned(soru)}")
    print("-" * 50)


📦 Çeviri kütüphanesi kuruluyor...
🔧 Çok Dilli RAG fonksiyonu oluşturuluyor...

🧪 Çok Dilli RAG Testi yapılıyor...

📨 Soru: İstanbulkart nedir ve nasıl alabilirim?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


💬 Cevap: İstanbulkart, İstanbul'daki tüm toplu taşıma araçlarında büyük indirimler sağlayan, yeniden yüklenebilir temassız bir akıllı karttır. Öğrenci indirimlerini aktif hale getirmek için herhangi bir İSTANBULKART başvuru merkezinde veya web sitesi/biyma uygulaması üzerinden öğrenci belgenizi/pasaportunuzu kullanarak kayıt yaptırmanız gerekmektedir. Kişiselleştirilmiş kartlarda fotoğrafınız bulunur ve transfer indirimlerine/aboneliklere olanak tanır. Portalı: https://www.istanbulkart.istanbul/
--------------------------------------------------

📨 Soru: How do I open a bank account in Turkey as a foreigner?
💬 Cevap: If you have a Turkish Foreigner ID (kimlik), tax number, and proof of residence, you can walk into any private bank branch with these documents to open an account. If you don't have a kimlik but have been resident for over a year, some branches may still consider opening a basic account.
--------------------------------------------------

📨 Soru: ما هي عقوبة تجاوز مدة التأ

In [6]:
# ======================================================
# 11. FASTAPI + NGROK API (COLAB UYUMLU KESİN ÇÖZÜM)
# ======================================================
print("\n🌐 API başlatılıyor...")

from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok, conf
import uvicorn
from uvicorn import Config, Server
import nest_asyncio
import subprocess
import time

# Colab'in asenkron döngü krizini çözen kilit nokta
nest_asyncio.apply()

app = FastAPI()

class ChatRequest(BaseModel):
    message: str
    history: list = []

@app.post("/chat")
async def chat_endpoint(req: ChatRequest):
    try:
        answer = rag_with_finetuned(req.message)
        return {"reply": answer, "source": "finetuned_qwen_rag_translated", "status": "success"}
    except Exception as e:
        return {"reply": f"Hata: {str(e)}", "source": "error", "status": "error"}

@app.get("/")
def root():
    return {
        "status": "online",
        "model": "Qwen2.5-7B-Finetuned-A100-Multilingual",
        "documents": collection.count(),
        "message": "E-YABANCI API Başarıyla Çalışıyor!"
    }

# Eski ngrok süreçlerini temizle
subprocess.run(["pkill", "-f", "ngrok"], capture_output=True)
time.sleep(2)

# DİKKAT: Ngrok token'ınızı eklediğinizden emin olun
NGROK_TOKEN = "3EdCXARFwle0UkAaeZZfeAM4PPS_5YLjwHiCSijdEsNx8U61y"
conf.get_default().auth_token = NGROK_TOKEN

try:
    public_url = ngrok.connect(8000)
    print(f"\n✅ Public URL: {public_url.public_url}")
    print(f"📱 Flutter'dan POST İsteği Atılacak Endpoint: {public_url.public_url}/chat")
    print("\n" + "=" * 60)
    print("🚀 API AYAĞA KALKTI! İstekleri Bekliyor...")
    print("=" * 60)

    # Colab için uyumlu asenkron sunucu başlatma
    config = Config(app, host="0.0.0.0", port=8000, loop="asyncio")
    server = Server(config)
    await server.serve()

except Exception as e:
    print(f"❌ Sunucu başlatılamadı: {e}")


🌐 API başlatılıyor...


INFO:     Started server process [8212]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



✅ Public URL: https://reabsorb-yanking-escapist.ngrok-free.dev
📱 Flutter'dan POST İsteği Atılacak Endpoint: https://reabsorb-yanking-escapist.ngrok-free.dev/chat

🚀 API AYAĞA KALKTI! İstekleri Bekliyor...


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [8212]
